# IMAGE TEXT EXTRACTION MODEL
## REALIX - Fake News Detection from Images

This notebook demonstrates the **Image → OCR → Fake News Detection** pipeline:
1. Load an image containing text (e.g., a screenshot of a news article)
2. Extract text using **Tesseract OCR**
3. Clean the extracted text using the same preprocessing as the text model
4. Run the text through the trained ISOT fake news detection ensemble
5. Return Real/Fake prediction with confidence

### Requirements
- **Tesseract OCR** must be installed on your system
  - Windows: Download from https://github.com/UB-Mannheim/tesseract/wiki
- Python packages: `pytesseract`, `Pillow`, `scikit-learn`, `xgboost`

In [ ]:
# ============================================================================
# STEP 0: INSTALL DEPENDENCIES
# ============================================================================
# Uncomment below if running in Colab or fresh environment
# !pip install pytesseract Pillow scikit-learn xgboost
# !apt-get install -y tesseract-ocr  # For Linux/Colab

import pytesseract
from PIL import Image, ImageFilter, ImageEnhance
import pickle
import re
import os
import numpy as np
import warnings
warnings.filterwarnings('ignore')

print("="*80)
print("IMAGE TEXT EXTRACTION MODEL - REALIX FAKE NEWS DETECTION")
print("="*80)

# ============================================================================
# CONFIGURE TESSERACT PATH (Windows)
# ============================================================================
# Update this path if Tesseract is installed elsewhere
TESSERACT_PATH = r'C:\Program Files\Tesseract-OCR\tesseract.exe'

if os.path.exists(TESSERACT_PATH):
    pytesseract.pytesseract.tesseract_cmd = TESSERACT_PATH
    print(f"✓ Tesseract found at: {TESSERACT_PATH}")
else:
    print(f"⚠ Tesseract not found at: {TESSERACT_PATH}")
    print("  Please install Tesseract OCR or update the path above.")
    print("  Download: https://github.com/UB-Mannheim/tesseract/wiki")

In [ ]:
# ============================================================================
# STEP 1: IMAGE PREPROCESSING FOR BETTER OCR
# ============================================================================

def preprocess_image_for_ocr(image):
    """
    Preprocess an image to improve OCR accuracy.
    - Convert to grayscale
    - Enhance contrast
    - Apply sharpening
    - Resize if too small
    """
    # Convert to grayscale
    gray = image.convert('L')
    
    # Enhance contrast
    enhancer = ImageEnhance.Contrast(gray)
    gray = enhancer.enhance(2.0)
    
    # Sharpen
    gray = gray.filter(ImageFilter.SHARPEN)
    
    # Resize if image is too small (OCR works better on larger images)
    width, height = gray.size
    if width < 1000:
        scale = 1000 / width
        new_size = (int(width * scale), int(height * scale))
        gray = gray.resize(new_size, Image.LANCZOS)
    
    return gray

print("✓ Image preprocessing function defined")

In [ ]:
# ============================================================================
# STEP 2: TEXT EXTRACTION USING TESSERACT OCR
# ============================================================================

def extract_text_from_image(image_path_or_pil):
    """
    Extract text from an image using Tesseract OCR.
    
    Args:
        image_path_or_pil: Either a file path (str) or a PIL Image object
    
    Returns:
        Extracted text string
    """
    # Load image
    if isinstance(image_path_or_pil, str):
        image = Image.open(image_path_or_pil)
    else:
        image = image_path_or_pil
    
    # Preprocess
    processed = preprocess_image_for_ocr(image)
    
    # OCR with custom config for better accuracy
    custom_config = r'--oem 3 --psm 6'
    text = pytesseract.image_to_string(processed, config=custom_config)
    
    # Clean up OCR artifacts
    text = text.strip()
    # Remove excessive whitespace/newlines
    text = re.sub(r'\n+', ' ', text)
    text = re.sub(r'\s+', ' ', text)
    
    return text

print("✓ Text extraction function defined")

In [ ]:
# ============================================================================
# STEP 3: TEXT CLEANING (SAME AS TEXT MODEL)
# ============================================================================

def clean_text(text):
    """
    Replicate the exact preprocessing from training notebook (clean_text_paper2):
    - Lowercase
    - Remove URLs
    - Remove HTML tags
    - Remove special characters and digits
    - Remove extra spaces
    """
    if not text or not isinstance(text, str):
        return ""
    text = text.lower()
    text = re.sub(r'https?://\S+|www\.\S+', '', text)
    text = re.sub(r'<.*?>', '', text)
    text = re.sub(r'[^a-z\s]', '', text)
    text = ' '.join(text.split())
    return text

print("✓ Text cleaning function defined (matching fknd_model preprocessing)")

In [ ]:
# ============================================================================
# STEP 4: LOAD TRAINED MODELS AND VECTORIZER
# ============================================================================

# Load models from backend directory
backend_dir = os.path.join(os.path.dirname(os.getcwd()), 'backend') \
    if 'backend' not in os.getcwd() else os.getcwd()

# Try multiple possible paths
possible_paths = [
    os.path.join('backend', 'vectorizer.pkl'),
    'vectorizer.pkl',
    os.path.join('..', 'backend', 'vectorizer.pkl'),
]

vectorizer = None
models = {}

for vpath in possible_paths:
    if os.path.exists(vpath):
        with open(vpath, 'rb') as f:
            vectorizer = pickle.load(f)
        model_path = vpath.replace('vectorizer.pkl', 'isot_trained_models.pkl')
        with open(model_path, 'rb') as f:
            trained_models = pickle.load(f)
        for name in trained_models:
            models[name] = trained_models[name]['model']
        print(f"✓ Loaded vectorizer from: {vpath}")
        print(f"✓ Loaded {len(models)} models: {list(models.keys())}")
        break

if vectorizer is None:
    print("⚠ Could not find model files. Make sure vectorizer.pkl and")
    print("  isot_trained_models.pkl are in the backend/ directory.")
    print("  Run fknd_model.ipynb first to generate these files.")

In [ ]:
# ============================================================================
# STEP 5: COMPLETE PIPELINE - IMAGE TO PREDICTION
# ============================================================================

def predict_from_image(image_path_or_pil):
    """
    Complete pipeline: Image → OCR → Clean → Predict
    
    Args:
        image_path_or_pil: File path string or PIL Image object
    
    Returns:
        dict with result, confidence, explanation, extracted_text
    """
    import time
    start_time = time.time()
    
    # Step 1: Extract text from image
    raw_text = extract_text_from_image(image_path_or_pil)
    print(f"\n📄 Extracted Text ({len(raw_text)} chars):")
    print(f"   '{raw_text[:200]}...'" if len(raw_text) > 200 else f"   '{raw_text}'")
    
    if len(raw_text.strip()) < 10:
        return {
            'result': 'Error',
            'confidence': 0.0,
            'explanation': 'Could not extract sufficient text from the image. Please upload a clearer image with readable text.',
            'extracted_text': raw_text,
            'analysis_time': f'{round(time.time() - start_time, 2)}s'
        }
    
    # Step 2: Clean extracted text
    cleaned_text = clean_text(raw_text)
    print(f"\n🧹 Cleaned Text: '{cleaned_text[:200]}...'" if len(cleaned_text) > 200 else f"\n🧹 Cleaned Text: '{cleaned_text}'")
    
    if len(cleaned_text.strip()) < 5:
        return {
            'result': 'Error',
            'confidence': 0.0,
            'explanation': 'Extracted text was too short after cleaning. Please upload an image with more readable text content.',
            'extracted_text': raw_text,
            'analysis_time': f'{round(time.time() - start_time, 2)}s'
        }
    
    # Step 3: Vectorize
    X = vectorizer.transform([cleaned_text])
    
    # Step 4: Ensemble prediction
    predictions = []
    confidences = []
    
    for name, model in models.items():
        pred = model.predict(X)[0]
        predictions.append(pred)
        
        if hasattr(model, 'predict_proba'):
            proba = model.predict_proba(X)[0]
            conf = float(proba[pred])
            confidences.append(conf)
        else:
            confidences.append(0.95)
        
        print(f"   {name}: {'FAKE' if pred == 0 else 'REAL'} (conf: {confidences[-1]:.4f})")
    
    # Majority vote
    fake_votes = sum(1 for p in predictions if p == 0)
    real_votes = sum(1 for p in predictions if p == 1)
    
    if fake_votes > real_votes:
        result = 'Fake'
        explanation = f'The ensemble of {len(models)} models ({', '.join(models.keys())}) classified the extracted text as FAKE.'
    else:
        result = 'Real'
        explanation = f'The ensemble of {len(models)} models ({', '.join(models.keys())}) classified the extracted text as REAL.'
    
    avg_confidence = float(np.mean(confidences))
    analysis_time = f'{round(time.time() - start_time, 2)}s'
    
    print(f"\n🎯 RESULT: {result} (Confidence: {avg_confidence*100:.1f}%)")
    print(f"⏱  Analysis Time: {analysis_time}")
    
    return {
        'result': result,
        'confidence': avg_confidence,
        'explanation': explanation,
        'extracted_text': raw_text,
        'analysis_time': analysis_time
    }

print("✓ Complete image prediction pipeline defined")

In [ ]:
# ============================================================================
# STEP 6: TEST THE PIPELINE
# ============================================================================

# Create a test image with text for demonstration
from PIL import ImageDraw, ImageFont

def create_test_image(text, filename='test_news_image.png'):
    """Create a test image with text for pipeline testing"""
    img = Image.new('RGB', (800, 400), color='white')
    draw = ImageDraw.Draw(img)
    
    # Use default font
    try:
        font = ImageFont.truetype('arial.ttf', 20)
    except:
        font = ImageFont.load_default()
    
    # Draw text with word wrapping
    words = text.split()
    lines = []
    current_line = ''
    for word in words:
        test_line = current_line + ' ' + word if current_line else word
        if len(test_line) > 70:
            lines.append(current_line)
            current_line = word
        else:
            current_line = test_line
    if current_line:
        lines.append(current_line)
    
    y = 30
    for line in lines:
        draw.text((30, y), line, fill='black', font=font)
        y += 30
    
    img.save(filename)
    print(f"✓ Test image saved: {filename}")
    return img

# Test with a sample news text
sample_news = (
    "Breaking News: Scientists at MIT have developed a new AI system "
    "that can detect fake news articles with over 95 percent accuracy. "
    "The research was published in the journal Nature and has been "
    "peer-reviewed by leading experts in the field of machine learning."
)

test_img = create_test_image(sample_news)

# Run the full pipeline on the test image
print("\n" + "="*80)
print("RUNNING PREDICTION PIPELINE ON TEST IMAGE")
print("="*80)

if vectorizer is not None:
    result = predict_from_image(test_img)
    print(f"\n📊 Final Result: {result}")
else:
    print("\n⚠ Cannot run prediction - models not loaded.")
    print("  Demonstrating OCR extraction only:")
    extracted = extract_text_from_image(test_img)
    print(f"  Extracted text: '{extracted}'")

## Summary

This Image Model provides:
1. **Image Preprocessing** - Grayscale conversion, contrast enhancement, sharpening, and resizing for optimal OCR
2. **OCR Text Extraction** - Using Tesseract with tuned configuration (`--oem 3 --psm 6`)
3. **Text Cleaning** - Same preprocessing pipeline as the text model (`clean_text_paper2`)
4. **Ensemble Prediction** - Uses all 3 trained models (Logistic Regression, Random Forest, XGBoost)
5. **Error Handling** - Graceful handling of images with no/insufficient text

The backend API `/predict-image` endpoint uses these same functions to serve predictions.